## Завдання 1: Прогнозування ризику раку молочної залози
- Мета:
    - передбачити, чи утворена пухлина є злоякісною або доброякісною.
- Дані:
    - розміри клітин,
    - форма,
    - текстура,
    - площа,
    - густина.
- Приклад датасету:
    - Breast Cancer Wisconsin (Diagnostic) Dataset
(https://www.kaggle.com/datasets/uciml/breast-cancer-wisconsin-data)

In [7]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler # Для масштабування даних
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score # Метрики для оцінки

import torch
import torch.nn as nn
import torch.optim as optim

try:
    df = pd.read_csv('HW-11.1_data.csv')  # наш дата сет    
    df = df.drop(columns=['id', 'Unnamed: 32'], errors='ignore')  # Видалимо непотрібні стовпці  
    df['diagnosis'] = df['diagnosis'].apply(lambda x: 1 if x == 'M' else 0)  # Перетворимо діагноз ('M'/'B') у числовий формат (1/0)

    # Визначаємо ознаки (X) та цільову змінну (y)
    y = df['diagnosis'].values  # наші ознаки
    X = df.drop('diagnosis', axis=1).values  # наша ціль

    # Розділимо дані: 70% на навчання, 30% на тимчасовий набір
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y) 
    
    # Розділимо тимчасовий набір порівну на валідаційний та тестовий (по 15% від загальної кількості)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)
        
    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Перетворимо дані у тензори PyTorch ---
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class CancerNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128), # Вхідний шар
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.3),

                nn.Linear(128, 64),       # Прихований шар
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.2),

                nn.Linear(64, 1)          # Вихідний шар 
            )

        def forward(self, x):
            return self.net(x)

    # Навчимо моделі 
    input_features = X_train.shape[1]
    model = CancerNN(input_dim=input_features)

    # Визначтмо функцію втрат та оптимізатор
    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)

    epochs = 150
    best_val_auc = 0.0
    best_state = None

    print("Почнемо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):        
        model.train()  # Режим навчання
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()  # Режим оцінки
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs  = torch.sigmoid(val_logits)
            val_auc    = roc_auc_score(y_val, val_probs.numpy())

        # Збережемо найкращу модель за показником AUC на валідаційній вибірці
        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 15 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")

    print("Навчання завершено.\n")

    # Оцінемо найкращі моделі на тестових даних 
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs  = torch.sigmoid(test_logits).numpy()
        test_preds  = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Доброякісна (B)', 'Злоякісна (M)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.1_data.csv' не знайдено.")
except Exception as e:
    print(f"Сталася помилка: {e}")

Почнемо навчання нейронної мережі...
Епоха 015 | Втрати на навчанні: 0.3085 | Val AUC: 0.994
Епоха 030 | Втрати на навчанні: 0.2207 | Val AUC: 0.995
Епоха 045 | Втрати на навчанні: 0.1653 | Val AUC: 0.995
Епоха 060 | Втрати на навчанні: 0.1267 | Val AUC: 0.996
Епоха 075 | Втрати на навчанні: 0.1044 | Val AUC: 0.996
Епоха 090 | Втрати на навчанні: 0.0772 | Val AUC: 0.996
Епоха 105 | Втрати на навчанні: 0.0646 | Val AUC: 0.996
Епоха 120 | Втрати на навчанні: 0.0554 | Val AUC: 0.995
Епоха 135 | Втрати на навчанні: 0.0421 | Val AUC: 0.995
Епоха 150 | Втрати на навчанні: 0.0452 | Val AUC: 0.995
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[54  0]
 [ 1 31]]

Детальний звіт:
                 precision    recall  f1-score   support

Доброякісна (B)      0.982     1.000     0.991        54
  Злоякісна (M)      1.000     0.969     0.984        32

       accuracy                          0.988        86
      macro avg      0.991     0.984     0.987        86
  

## Завдання 2: Прогнозування рівня виживання пацієнтів з раком легенів
- Мета:
    - передбачити, чи пацієнт виживе протягом 5 років після діагнозу.
- Дані:
  - вік,
  - стать,
  - стадія хвороби,
  - метод лікування,
  - результати обстежень.
- Приклад датасету:
    - Lung Cancer Data (https://www.kaggle.com/datasets/shreyasparaj1/lung-cancer-dataset)

In [10]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim

# Завантажимо дані
try:
    df = pd.read_csv('HW-11.2_Lung_Cancer_Dataset.csv')

    # Перетворимо категоріальні текстові дані у числові: 1, 0   
    df['GENDER'] = df['GENDER'].apply(lambda x: 1 if x == 'M' else 0)
    df['LUNG_CANCER'] = df['LUNG_CANCER'].apply(lambda x: 1 if x == 'YES' else 0)

    # Визначимо ознаки (X) та цільову змінну (y)
    y = df['LUNG_CANCER'].values
    X = df.drop('LUNG_CANCER', axis=1).values

    # Розділимо дані: 70% на навчання, 30% на тимчасовий набір
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        
    # Розділимо тимчасовий набір на валідаційний та тестовий (по 15%)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Перетворимо дані у тензори PyTorch ---
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class LungCancerNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            return self.net(x)

    # Навчимо моделі
    input_features = X_train.shape[1]
    model = LungCancerNN(input_dim=input_features)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 100
    best_val_auc = 0.0
    best_state = None

    print("Почнемо навчання нейронної мережі для прогнозування раку легенів...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs  = torch.sigmoid(val_logits)
            val_auc    = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 10 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")

    print("Навчання завершено.\n")

    # Оцінемо найкращі моделі на тестових даних
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs  = torch.sigmoid(test_logits).numpy()
        test_preds  = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Немає раку (NO)', 'Є рак (YES)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.2_Lung_Cancer_Dataset.csv' не знайдено.")
except Exception as e:
    print(f"Сталася помилка: {e}")

Почнемо навчання нейронної мережі для прогнозування раку легенів...
Епоха 010 | Втрати на навчанні: 0.5369 | Val AUC: 0.363
Епоха 020 | Втрати на навчанні: 0.4561 | Val AUC: 0.371
Епоха 030 | Втрати на навчанні: 0.3815 | Val AUC: 0.433
Епоха 040 | Втрати на навчанні: 0.3289 | Val AUC: 0.542
Епоха 050 | Втрати на навчанні: 0.2886 | Val AUC: 0.642
Епоха 060 | Втрати на навчанні: 0.2694 | Val AUC: 0.792
Епоха 070 | Втрати на навчанні: 0.2301 | Val AUC: 0.829
Епоха 080 | Втрати на навчанні: 0.2233 | Val AUC: 0.863
Епоха 090 | Втрати на навчанні: 0.2141 | Val AUC: 0.892
Епоха 100 | Втрати на навчанні: 0.1933 | Val AUC: 0.900
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[ 1  5]
 [ 1 40]]

Детальний звіт:
                 precision    recall  f1-score   support

Немає раку (NO)      0.500     0.167     0.250         6
    Є рак (YES)      0.889     0.976     0.930        41

       accuracy                          0.872        47
      macro avg      0.694  

## Завдання 3: Прогнозування ризику діабету
- Мета:
    - визначити ймовірність діабету у пацієнта.
- Дані:
    - вік,
    - ІМТ,
    - рівень глюкози,
    - кров’яний тиск,
    - сімейна історія.
- Приклад датасету:
    - Pima Indians Diabetes Dataset (https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim

try:
     
    df = pd.read_csv('HW-11.3_diabetes.csv') # Завантажимо датасет

    # Визначимо ознаки (X) та цільову змінну (y)
    X = df.drop('Outcome', axis=1).values
    y = df['Outcome'].values

    # Розділимо дані: 70% на навчання, 30% на тимчасовий набір
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
        
    # Розділимо тимчасовий набір на валідаційний та тестовий (по 15%)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Перетворимо дані у тензори PyTorch
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class DiabetesNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(32, 1)
            )

        def forward(self, x):
            return self.net(x)

    # Навчимо моделі
    input_features = X_train.shape[1]
    model = DiabetesNN(input_dim=input_features)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 150
    best_val_auc = 0.0
    best_state = None

    print("Починаємо навчання нейронної мережі для прогнозування діабету...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs  = torch.sigmoid(val_logits)
            val_auc    = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 15 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")

    print("Навчання завершено.\n")

    # Оцінемо найкращі моделі на тестових даних
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs  = torch.sigmoid(test_logits).numpy()
        test_preds  = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Немає діабету (0)', 'Є діабет (1)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.3_diabetes.csv' не знайдено.")
except Exception as e:
    print(f"Сталася помилка: {e}")

Починаємо навчання нейронної мережі для прогнозування діабету...
Епоха 015 | Втрати на навчанні: 0.6718 | Val AUC: 0.675
Епоха 030 | Втрати на навчанні: 0.6046 | Val AUC: 0.760
Епоха 045 | Втрати на навчанні: 0.5378 | Val AUC: 0.792
Епоха 060 | Втрати на навчанні: 0.5080 | Val AUC: 0.806
Епоха 075 | Втрати на навчанні: 0.4655 | Val AUC: 0.808
Епоха 090 | Втрати на навчанні: 0.4688 | Val AUC: 0.813
Епоха 105 | Втрати на навчанні: 0.4635 | Val AUC: 0.818
Епоха 120 | Втрати на навчанні: 0.4433 | Val AUC: 0.816
Епоха 135 | Втрати на навчанні: 0.4448 | Val AUC: 0.816
Епоха 150 | Втрати на навчанні: 0.4292 | Val AUC: 0.815
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[66  9]
 [20 21]]

Детальний звіт:
                   precision    recall  f1-score   support

Немає діабету (0)      0.767     0.880     0.820        75
     Є діабет (1)      0.700     0.512     0.592        41

         accuracy                          0.750       116
        macro avg      

## Завдання 4: Прогнозування дефолту кредиту
- Мета:
    - передбачити, чи клієнт не зможе сплатити кредит.
- Дані:
    - дохід,
    - вік,
    - кредитна історія,
    - сума кредиту,
    - тип роботи.
- Приклад датасету:
    - Give Me Some Credit (https://www.kaggle.com/c/GiveMeSomeCredit)

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

import torch
import torch.nn as nn
import torch.optim as optim

try:
    # Завантажимо дані
    df = pd.read_csv('HW-11.4_Give Me Some Credit_cs-training.csv')

    # Видалимо перший стовпець, який є просто індексом
    if 'Unnamed: 0' in df.columns:
        df = df.drop(columns=['Unnamed: 0'])

    # Заповнимо пропуски в доході медіанним значенням
    if df['MonthlyIncome'].isnull().any():
        median_income = df['MonthlyIncome'].median()
        df['MonthlyIncome'] = df['MonthlyIncome'].fillna(median_income)
        print(f"Пропущені значення в 'MonthlyIncome' заповнено медіаною: {median_income}")

    # Заповнимо пропуски в кількості утриманців медіанним значенням
    if df['NumberOfDependents'].isnull().any():
        median_dependents = df['NumberOfDependents'].median()
        df['NumberOfDependents'] = df['NumberOfDependents'].fillna(median_dependents)
        print(f"Пропущені значення в 'NumberOfDependents' заповнено медіаною: {median_dependents}\n")

    # Визначимо ознаки (X) та цільову змінну (y)
    y = df['SeriousDlqin2yrs'].values
    X = df.drop('SeriousDlqin2yrs', axis=1).values

    # Розділимо дані з урахуванням стратифікації для збереження балансу класів
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val   = scaler.transform(X_val)
    X_test  = scaler.transform(X_test)

    # Перетворимо дані у тензори PyTorch
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t   = torch.tensor(X_val,   dtype=torch.float32)
    y_val_t   = torch.tensor(y_val,   dtype=torch.float32).unsqueeze(1)
    X_test_t  = torch.tensor(X_test,  dtype=torch.float32)
    y_test_t  = torch.tensor(y_test,  dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class CreditDefaultNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 128),
                nn.BatchNorm1d(128),
                nn.ReLU(),
                nn.Dropout(0.4),

                nn.Linear(128, 64),
                nn.BatchNorm1d(64),
                nn.ReLU(),
                nn.Dropout(0.3),

                nn.Linear(64, 1)
            )

        def forward(self, x):
            return self.net(x)

    # Навчимо моделі
    input_features = X_train.shape[1]
    model = CreditDefaultNN(input_dim=input_features)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 100
    best_val_auc = 0.0
    best_state = None

    print("Починаємо навчання нейронної мережі для прогнозування дефолту...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs  = torch.sigmoid(val_logits)
            val_auc    = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 10 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")
    print("Навчання завершено.\n")

    # Оцінемо найкращі моделі на тестових даних
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs  = torch.sigmoid(test_logits).numpy()
        test_preds  = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Не було дефолту (0)', 'Був дефолт (1)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.4_Give Me Some Credit_cs-training.csv' не знайдено.")
except Exception as e:
    print(f"Сталася помилка: {e}")

Пропущені значення в 'MonthlyIncome' заповнено медіаною: 5400.0
Пропущені значення в 'NumberOfDependents' заповнено медіаною: 0.0

Починаємо навчання нейронної мережі для прогнозування дефолту...
Епоха 010 | Втрати на навчанні: 0.6009 | Val AUC: 0.638
Епоха 020 | Втрати на навчанні: 0.4950 | Val AUC: 0.607
Епоха 030 | Втрати на навчанні: 0.4304 | Val AUC: 0.585
Епоха 040 | Втрати на навчанні: 0.3845 | Val AUC: 0.586
Епоха 050 | Втрати на навчанні: 0.3458 | Val AUC: 0.604
Епоха 060 | Втрати на навчанні: 0.3167 | Val AUC: 0.629
Епоха 070 | Втрати на навчанні: 0.2955 | Val AUC: 0.659
Епоха 080 | Втрати на навчанні: 0.2780 | Val AUC: 0.690
Епоха 090 | Втрати на навчанні: 0.2649 | Val AUC: 0.715
Епоха 100 | Втрати на навчанні: 0.2551 | Val AUC: 0.736
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[20965    31]
 [ 1480    24]]

Детальний звіт:
                     precision    recall  f1-score   support

Не було дефолту (0)      0.934     0.999     0.965     2

## Завдання 5: Прогнозування підвищення продажів у магазині
- Мета: передбачити, чи клієнт зробить покупку наступного тижня.
- Дані: 
    - історія покупок, 
    - демографія, 
    - кількість відвідувань сайту.
- Приклад датасету: Online Retail Dataset (https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)

In [3]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim
import warnings

warnings.filterwarnings('ignore')

try:
    # Завантажимо та об'єднаємо дані
    print("Завантаження даних...")
    customers = pd.read_csv('HW-11.5_olist_customers_dataset.csv')
    orders = pd.read_csv('HW-11.5_olist_orders_dataset.csv')
    items = pd.read_csv('HW-11.5_olist_order_items_dataset.csv')

    print("Об'єднаємо таблиці...")    
    df = pd.merge(orders, customers, on='customer_id')  # Об'єднуємо замовлення з клієнтами    
    df = pd.merge(df, items, on='order_id')             # Додаємо інформацію про товари в замовленнях
    print("Дані успішно об'єднано.")

    # Підготовимо дані та створимо ознаки (Feature Engineering)
    print("Підготовка даних та створення ознак...")
    df['order_purchase_timestamp'] = pd.to_datetime(df['order_purchase_timestamp'])

    # Для аналізу беремо тільки доставлені замовлення
    df = df[df['order_status'] == 'delivered'].copy()

    # Розрахуємо повну вартість кожного товару в замовленні
    df['total_value'] = df['price'] + df['freight_value']

    # Визначимо "сьогоднішній день" як останню дату в датасеті + 1 день
    snapshot_date = df['order_purchase_timestamp'].max() + pd.Timedelta(days=1)

    # Групуємо дані по кожному унікальному клієнту
    rfm_data = df.groupby('customer_unique_id').agg({
        'order_purchase_timestamp': lambda date: (snapshot_date - date.max()).days,
        'order_id': 'count',
        'total_value': 'sum'
    })

    # Перейменуємо стовпці для зрозумілості
    rfm_data.rename(columns={'order_purchase_timestamp': 'Recency',
                             'order_id': 'Frequency',
                             'total_value': 'Monetary'}, inplace=True)

    # --- Крок 3: Створення цільової змінної ---
    # Визначаємо "активних" клієнтів (робили покупку за останні 60 днів). Це наша цільова змінна y
    rfm_data['Is_Active'] = (rfm_data['Recency'] <= 60).astype(int)
    print("Ознаки (Recency, Frequency, Monetary) та цільова змінна створені.")

    # Визначимо ознаки (X) та цільову змінну (y)
    y = rfm_data['Is_Active'].values
    X = rfm_data.drop(['Is_Active'], axis=1).values

    # Розділяємо дані
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Перетворимо в тензори
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class SalesNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, 1)
            )
        def forward(self, x):
            return self.net(x)

    input_features = X_train.shape[1]
    model = SalesNN(input_dim=input_features)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 100
    best_val_auc = 0.0
    best_state = None

    print("\nПочинемо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits)
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 10 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")

    print("Навчання завершено.\n")

    # --- Крок 5: Оцінка найкращої моделі ---
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs = torch.sigmoid(test_logits).numpy()
        test_preds = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Неактивний (0)', 'Активний (1)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError as e:
    print(f"Помилка: Файл не знайдено. Переконайтеся, що всі три файли знаходяться в одній папці: {e.filename}")
except Exception as e:
    print(f"Сталася неочікувана помилка: {e}")

Завантаження даних...
Об'єднаємо таблиці...
Дані успішно об'єднано.
Підготовка даних та створення ознак...
Ознаки (Recency, Frequency, Monetary) та цільова змінна створені.

Починемо навчання нейронної мережі...
Епоха 010 | Втрати на навчанні: 0.6644 | Val AUC: 0.184
Епоха 020 | Втрати на навчанні: 0.5686 | Val AUC: 0.310
Епоха 030 | Втрати на навчанні: 0.4829 | Val AUC: 0.526
Епоха 040 | Втрати на навчанні: 0.4111 | Val AUC: 0.854
Епоха 050 | Втрати на навчанні: 0.3500 | Val AUC: 0.948
Епоха 060 | Втрати на навчанні: 0.2976 | Val AUC: 0.980
Епоха 070 | Втрати на навчанні: 0.2561 | Val AUC: 0.991
Епоха 080 | Втрати на навчанні: 0.2231 | Val AUC: 0.994
Епоха 090 | Втрати на навчанні: 0.1966 | Val AUC: 0.996
Епоха 100 | Втрати на навчанні: 0.1759 | Val AUC: 0.997
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[12102    37]
 [  212  1653]]

Детальний звіт:
                precision    recall  f1-score   support

Неактивний (0)      0.983     0.997     0.990

## Завдання 6: Прогнозування винагороди співробітників
- Мета: передбачити, чи співробітник отримає підвищення або бонус.
- Дані:
    - вік,
    - досвід роботи,
    - продуктивність,
    - оцінка керівництва.
- Приклад датасету: Employee Promotion Dataset (https://www.kaggle.com/datasets/aminaidhm23386/employee-worker)

In [5]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

try:
    print("Завантаження даних...")
    df = pd.read_csv('HW-11.6_HR.csv')
    
    print("Обробка категоріальних даних...")
    # Обробимо порядкову ознаку, рівень 'salary': 'low', 'medium', 'high' замінимо на числа (0, 1, 2)
    salary_map = {'low': 0, 'medium': 1, 'high': 2}
    df['salary'] = df['salary'].map(salary_map)

    # Обробимо порядкову ознаку 'sales' (департамент) за допомогою One-Hot Encoding
    df = pd.get_dummies(df, columns=['sales'], drop_first=True)  # створимо окремий стовпець (0 або 1) для кожного департаменту
    print("Дані успішно перетворено.")

    # Визначимо ознаки (X) та цільову змінну (y)
    y = df['promotion_last_5years'].values
    X = df.drop('promotion_last_5years', axis=1).values

    # Розділимо дані зі стратифікацією через сильний дисбаланс класів
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Перетворимо в тензори
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # Створимо архітектуру нейронної мережі
    class HR_NN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(32, 1)
            )
        def forward(self, x):
            return self.net(x)

    input_features = X_train.shape[1]
    model = HR_NN(input_dim=input_features)

    # Визначмсо ваги для позитивного класу для боротьби з дисбалансом
    pos_weight = torch.tensor([sum(y_train==0) / sum(y_train==1)])
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 70
    best_val_auc = 0.0
    best_state = None

    print("\nПочнемо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits)
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 10 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")
    print("Навчання завершено.\n")

    # Оцінемо найкращу модель
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs = torch.sigmoid(test_logits).numpy()
        test_preds = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:")
    print("Зверніть увагу на показники для класу 'Було підвищення', оскільки він є рідкісним.")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Не було підвищення (0)', 'Було підвищення (1)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.6_HR.csv' не знайдено.")
except Exception as e:
    print(f"Сталася неочікувана помилка: {e}")

Завантаження даних...
Обробка категоріальних даних...
Дані успішно перетворено.

Почнемо навчання нейронної мережі...
Епоха 010 | Втрати на навчанні: 1.3138 | Val AUC: 0.679
Епоха 020 | Втрати на навчанні: 1.2668 | Val AUC: 0.699
Епоха 030 | Втрати на навчанні: 1.2022 | Val AUC: 0.701
Епоха 040 | Втрати на навчанні: 1.1352 | Val AUC: 0.702
Епоха 050 | Втрати на навчанні: 1.0867 | Val AUC: 0.704
Епоха 060 | Втрати на навчанні: 1.0589 | Val AUC: 0.708
Епоха 070 | Втрати на навчанні: 1.0342 | Val AUC: 0.713
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[1583  619]
 [  13   35]]

Детальний звіт:
Зверніть увагу на показники для класу 'Було підвищення', оскільки він є рідкісним.
                        precision    recall  f1-score   support

Не було підвищення (0)      0.992     0.719     0.834      2202
   Було підвищення (1)      0.054     0.729     0.100        48

              accuracy                          0.719      2250
             macro avg     

## Завдання 7: Прогнозування рівня ризику страхування
- Мета: передбачити, чи клієнт буде високо ризиковим для страхування.
- Дані:
    - вік,
    - історія страхових випадків,
    - стан здоров’я,
    - робота,
    - доходи.
- Приклад датасету: Insurance Risk Dataset (https://www.kaggle.com/code/shivamb/deep-healthcare-analysis-using-bigquery/input)

In [12]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

print("Завантаження даних...")
df = pd.read_csv('HW-11.7_Prudential Life Insurance Assessment.csv')
print("Рядки, стовпчики:", df.shape)
df = df.copy() # дефрагментація

# видалимо технічний Id
if 'Id' in df.columns:
    df = df.drop('Id', axis=1)

print("Підрахунок значень відповідей:\n", df['Response'].value_counts().sort_index())

# Об'єднаємо всі колонки з назвами 'Medical_Keyword' в одну 'medical_kw_count'
med_cols = [c for c in df.columns if c.startswith('Medical_Keyword_')]
if med_cols:
    df['medical_kw_count'] = df[med_cols].sum(axis=1)

# Перетворимо словесну колонку 'Product_Info_2' в числову колонку 'Product_Info_2_enc'
if 'Product_Info_2' in df.columns:
    le = LabelEncoder()
    df['Product_Info_2_enc'] = le.fit_transform(df['Product_Info_2'].astype(str))
    df = df.drop('Product_Info_2', axis=1)

num_cols = df.select_dtypes(include=[np.number]).columns.tolist() # колонки з числами об'єднаємо в одну колонку 'num_cols'
num_cols = [c for c in num_cols if c != 'Response']  # колонку Response, що ми хочемо передбачити, виключаємо
df[num_cols] = df[num_cols].fillna(df[num_cols].median())  # заповнимо пусті колонки середнім числом

threshold = 5  # вибиремо поріг: >=5 як high-risk
high_risk = (df['Response'] >= threshold).astype(int)  # перетворюємо числа в так/ні: «це високий ризик чи ні»
df = pd.concat([df, high_risk.rename('high_risk')], axis=1)

# Визначимо ознаки (X) та цільову змінну (y)
X = df.drop(columns=['high_risk'])
y = df['high_risk'].values

# Переведемо всі словесні колонки в числові 
for c in X.select_dtypes(include=['object']).columns:
    X[c] = LabelEncoder().fit_transform(X[c].astype(str))

# Масштабуємо дані
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X.values)

# Розділимо дані зі стратифікацією через сильний дисбаланс класів
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

# Перетворимо в тензори
X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_val_t = torch.tensor(X_val, dtype=torch.float32)
y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# Створимо модель та навчмсо нейронну мережу
class SimpleNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1) 
        )
    def forward(self, x):
        return self.net(x)

# Навчимо моделі
input_dim = X_train.shape[1]
model = SimpleNN(input_dim)

# Обчислимо втрати та оптимізатор (баланс класів)
pos = (y_train == 1).sum()
neg = (y_train == 0).sum()
print(f"Train pos: {pos}, neg: {neg}")
if pos == 0:
    pos_weight = torch.tensor([1.0])
else:
    pos_weight = torch.tensor([neg / pos], dtype=torch.float32)

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# Тренування нейронної мережі
epochs = 50
best_val_auc = 0.0
best_state = None

print("\nПочинаємо навчання нейронної мережі для прогнозування рівня ризику страхування...")
for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    logits = model(X_train_t)
    loss = criterion(logits, y_train_t)
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_logits = model(X_val_t)
        val_probs = torch.sigmoid(val_logits).numpy()
        try:
            val_auc = roc_auc_score(y_val, val_probs)
        except Exception:
            val_auc = 0.0

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = model.state_dict()

    if epoch % 10 == 0 or epoch == 1:
        print(f"Epoch {epoch:02d} | loss: {loss.item():.4f} | val_auc: {val_auc:.4f}")
print("Навчання завершено. Найліпший показник AUC:", best_val_auc)

# Оцінемо найкращі моделі на тестових даних
if best_state is not None:
    model.load_state_dict(best_state)
model.eval()
with torch.no_grad():
    test_logits = model(X_test_t)
    test_probs = torch.sigmoid(test_logits).numpy().ravel()
    test_preds = (test_probs > 0.5).astype(int)

print("\nРезультати оцінки на тестових даних:")
print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
print("\nДетальний звіт:")
print(classification_report(y_test, test_preds, digits=3))
try:
    print("ROC-AUC:", roc_auc_score(y_test, test_probs))
except:
    pass

Завантаження даних...
Рядки, стовпчики: (59381, 128)
Підрахунок значень відповідей:
 Response
1     6207
2     6552
3     1013
4     1428
5     5432
6    11233
7     8027
8    19489
Name: count, dtype: int64
Train pos: 30926, neg: 10640

Починаємо навчання нейронної мережі для прогнозування рівня ризику страхування...
Epoch 01 | loss: 0.3594 | val_auc: 0.5163
Epoch 10 | loss: 0.3269 | val_auc: 0.8027
Epoch 20 | loss: 0.2839 | val_auc: 0.8422
Epoch 30 | loss: 0.2381 | val_auc: 0.8864
Epoch 40 | loss: 0.1909 | val_auc: 0.9348
Epoch 50 | loss: 0.1436 | val_auc: 0.9719
Навчання завершено. Найліпший показник AUC: 0.9719316776927984

Результати оцінки на тестових даних:
Матриця плутанини:
 [[2136  144]
 [ 742 5886]]

Детальний звіт:
              precision    recall  f1-score   support

           0      0.742     0.937     0.828      2280
           1      0.976     0.888     0.930      6628

    accuracy                          0.901      8908
   macro avg      0.859     0.912     0.879  

## Завдання 8: Класифікація видів тварин по характеристикам
- Мета: передбачити вид тварини (бінарно чи мультиклас).
- Дані:
    - вага,
    - зріст,
    - довжина хвоста,
    - кількість ніг,
    - тип харчування.
- Приклад датасету: Zoo Dataset (https://www.kaggle.com/datasets/uciml/zoo-animal-classification/data)

In [8]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
import torch
import torch.nn as nn
import torch.optim as optim

try:
    print("Завантаження даних...")
    df = pd.read_csv('HW-11.8_zoo.csv')   
    
    print("Пошук та фільтрація рідкісних класів...")    
    class_counts = df['class_type'].value_counts()   # Рахуємо кількість екземплярів у кожному класі    
    rare_classes = class_counts[class_counts < 2].index  # Визначаємо класи, де менше 2-х екземплярів (мінімум для поділу)

    if not rare_classes.empty:        
        df = df[~df['class_type'].isin(rare_classes)]  # Видаляємо рядки, що належать до рідкісних класів
        print(f"Було видалено класи з одним екземпляром: {list(rare_classes)}")
    else:
        print("Рідкісних класів (менше 2 екземплярів) не знайдено.\n")
        
        # Підготуємо дані 
    # Визначимо ознаки (X) та цільову змінну (y)
    y_raw = df['class_type'].values
    X = df.drop(['class_type', 'animal_name'], axis=1).values

    # Перекодуємо мітки класів, щоб вони йшли послідовно (0, 1, 2...)    
    le = LabelEncoder()
    y = le.fit_transform(y_raw)
    
    # Збережемо відповідність нових міток до старих для звіту
    class_names = [str(c) for c in le.classes_]
    print(f"Класи, що залишились для навчання: {class_names}\n")
    
            # Підготуємо дані до навчання
    # Розділимо дані зі стратифікацією
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.4, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Перетворимо в тензори. 
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.long) # !!!Для CrossEntropyLoss потрібен тип Long!!!
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.long)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.long)

    
            # Створимо та навчимо нейронну мережу
    # Визначаємо кількість класів для вихідного шару
    num_classes = len(np.unique(y))

    class ZooNN(nn.Module):
        def __init__(self, input_dim, output_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(),
                nn.Linear(64, 32),
                nn.ReLU(),
                nn.Linear(32, output_dim) # Вихідний шар з нейронами по числу класів
            )
        def forward(self, x):
            return self.net(x)

    input_features = X_train.shape[1]
    model = ZooNN(input_dim=input_features, output_dim=num_classes)

    # Використаємо CrossEntropyLoss для багатокласової класифікації
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.005)

    epochs = 150
    best_val_acc = 0.0
    best_state = None

    print("\nПочинаємо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        # Оцінемо
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)           
            val_preds = torch.argmax(val_logits, dim=1)  # Знаходимо клас з найвищою ймовірністю
            val_acc = accuracy_score(y_val_t.numpy(), val_preds.numpy())

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = model.state_dict()

        if epoch % 15 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val Accuracy: {val_acc:.2%}")
    print("Навчання завершено.\n")

            # Оцінемо найкращі моделі
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_preds = torch.argmax(test_logits, dim=1).numpy()

    print("Результати оцінки на тестових даних:")
    print(f"Загальна точність (Accuracy): {accuracy_score(y_test, test_preds):.2%}\n")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт по класах:")
    print(classification_report(y_test, test_preds, digits=3))

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.8_zoo.csv' не знайдено.")
except Exception as e:
    print(f"Сталася неочікувана помилка: {e}")

Завантаження даних...
Пошук та фільтрація рідкісних класів...
Рідкісних класів (менше 2 екземплярів) не знайдено.

Класи, що залишились для навчання: ['1', '2', '3', '4', '5', '6', '7']


Починаємо навчання нейронної мережі...
Епоха 015 | Втрати на навчанні: 0.5780 | Val Accuracy: 90.00%
Епоха 030 | Втрати на навчанні: 0.1056 | Val Accuracy: 90.00%
Епоха 045 | Втрати на навчанні: 0.0223 | Val Accuracy: 95.00%
Епоха 060 | Втрати на навчанні: 0.0052 | Val Accuracy: 95.00%
Епоха 075 | Втрати на навчанні: 0.0019 | Val Accuracy: 95.00%
Епоха 090 | Втрати на навчанні: 0.0011 | Val Accuracy: 95.00%
Епоха 105 | Втрати на навчанні: 0.0008 | Val Accuracy: 95.00%
Епоха 120 | Втрати на навчанні: 0.0006 | Val Accuracy: 95.00%
Епоха 135 | Втрати на навчанні: 0.0005 | Val Accuracy: 95.00%
Епоха 150 | Втрати на навчанні: 0.0004 | Val Accuracy: 95.00%
Навчання завершено.

Результати оцінки на тестових даних:
Загальна точність (Accuracy): 100.00%

Матриця плутанини:
 [[9 0 0 0 0 0 0]
 [0 4 0 0 0 0 0]
 [

## Завдання 9: Прогнозування нещасних випадків на виробництві
- Мета: передбачити, чи відбудеться інцидент.
- Дані:
    - тип роботи,
    - стаж,
    - умови безпеки,
    - кількість співробітників.
- Приклад датасету: Accidents Dataset (https://www.kaggle.com/datasets/ksbmishra/employeeturnoverprediction/data)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

try:
    print("Завантаження даних...")
    df = pd.read_csv('HW-11.9_EmployeeData.csv')
    
    print("Обробимо категоріальні дані...")
    # Перетворимо 'salary' (low: 0, medium: 1, high: 2)
    salary_map = {'low': 0, 'medium': 1, 'high': 2}
    df['salary'] = df['salary'].map(salary_map)

    # Перетворимо 'sales' за допомогою One-Hot Encoding
    df = pd.get_dummies(df, columns=['sales'], drop_first=True)
    print("Дані успішно перетворено.")

    # --- Підготовка до навчання ---
    # Визначимо ознаки (X) та цільову змінну (y)
    y = df['Work_accident'].values   # Цільова змінна - 'Work_accident'
    X = df.drop('Work_accident', axis=1).values

    # Розділяємо дані у пропорції 70/15/15 зі стратифікацією
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

    # Масштабуємо дані
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_val = scaler.transform(X_val)
    X_test = scaler.transform(X_test)

    # Перетворимо в тензори.
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # Створимо та навчимо нейронну мережу
    class AccidentNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64), 
                nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32), 
                nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(32, 1)
            )
        def forward(self, x):
            return self.net(x)

    input_features = X_train.shape[1]
    model = AccidentNN(input_dim=input_features)

    # Розрахунок ваги для позитивного класу (інцидент) для боротьби з дисбалансом
    if sum(y_train==1) > 0:
        pos_weight = torch.tensor([sum(y_train==0) / sum(y_train==1)])
    else:
        pos_weight = torch.tensor([1.0])
        
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    epochs = 70
    best_val_auc = 0.0
    best_state = None

    print("\nПочинемо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits)
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 10 == 0:
            print(f"Епоха {epoch:03d} | Втрати на навчанні: {loss.item():.4f} | Val AUC: {val_auc:.3f}")

    print("Навчання завершено.\n")

    # Оцінемо найкращі моделі
    model.load_state_dict(best_state)
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs = torch.sigmoid(test_logits).numpy()
        test_preds = (test_probs > 0.5).astype(int)

    print("Результати оцінки на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт по класах:")
    print(classification_report(y_test, test_preds, digits=3, target_names=['Без інцидентів (0)', 'Був інцидент (1)']))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.3f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.9_EmployeeData.csv' не знайдено.")
except Exception as e:
    print(f"Сталася неочікувана помилка: {e}")

Завантаження даних...
Обробимо категоріальні дані...
Дані успішно перетворено.

Починемо навчання нейронної мережі...
Епоха 010 | Втрати на навчанні: 1.1822 | Val AUC: 0.557
Епоха 020 | Втрати на навчанні: 1.1703 | Val AUC: 0.586
Епоха 030 | Втрати на навчанні: 1.1576 | Val AUC: 0.587
Епоха 040 | Втрати на навчанні: 1.1413 | Val AUC: 0.587
Епоха 050 | Втрати на навчанні: 1.1320 | Val AUC: 0.589
Епоха 060 | Втрати на навчанні: 1.1268 | Val AUC: 0.594
Епоха 070 | Втрати на навчанні: 1.1221 | Val AUC: 0.600
Навчання завершено.

Результати оцінки на тестових даних:
Матриця плутанини:
 [[ 609 1316]
 [  40  285]]

Детальний звіт по класах:
                    precision    recall  f1-score   support

Без інцидентів (0)      0.938     0.316     0.473      1925
  Був інцидент (1)      0.178     0.877     0.296       325

          accuracy                          0.397      2250
         macro avg      0.558     0.597     0.385      2250
      weighted avg      0.829     0.397     0.448      2

## Завдання 9: Прогнозування кредитного ризику на основі транзакцій
- Мета: визначити, чи клієнт буде ризиковим на основі фінансової активності.
- Дані:
    - кількість транзакцій,
    - сума,
    - частота прострочень,
    - доходи.
- Приклад датасету: Credit Card Fraud Detection (https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import torch
import torch.nn as nn
import torch.optim as optim

try:
    print("Завантаження даних...")
    df = pd.read_csv("HW-11.10_creditcard.csv")

    print("Попередня обробка даних...")

    # --- Підготовка до навчання ---
    # Визначимо ознаки (X) та цільову змінну (y)
    y = df["Class"].values
    X = df.drop("Class", axis=1).copy()

    # Лог-трансформація суми
    X["Amount"] = np.log1p(X["Amount"])

    # Масштабуємо дані
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)

    # Розділяємо дані у пропорції 70/15/15 зі стратифікацією
    X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y, test_size=0.3, stratify=y, random_state=42)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=42)

    # --- Перетворення в тензори ---
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
    X_val_t = torch.tensor(X_val, dtype=torch.float32)
    y_val_t = torch.tensor(y_val, dtype=torch.float32).unsqueeze(1)
    X_test_t = torch.tensor(X_test, dtype=torch.float32)
    y_test_t = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

    # Створимо та навчимо нейронну мережу
    class FraudNN(nn.Module):
        def __init__(self, input_dim):
            super().__init__()
            self.net = nn.Sequential(
                nn.Linear(input_dim, 64),
                nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(64, 32),
                nn.ReLU(), nn.Dropout(0.2),
                nn.Linear(32, 1)
            )
        def forward(self, x):
            return self.net(x)

    input_features = X_train.shape[1]
    model = FraudNN(input_dim=input_features)

    # Балансування класів
    pos_weight = torch.tensor([sum(y_train==0) / sum(y_train==1)], dtype=torch.float32)
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # --- Навчання ---
    epochs = 50
    best_val_auc = 0.0
    best_state = None

    print("\nПочинаємо навчання нейронної мережі...")
    for epoch in range(1, epochs + 1):
        model.train()
        optimizer.zero_grad()
        logits = model(X_train_t)
        loss = criterion(logits, y_train_t)
        loss.backward()
        optimizer.step()

        # Валідація
        model.eval()
        with torch.no_grad():
            val_logits = model(X_val_t)
            val_probs = torch.sigmoid(val_logits)
            val_auc = roc_auc_score(y_val_t.numpy(), val_probs.numpy())

        if val_auc > best_val_auc:
            best_val_auc = val_auc
            best_state = model.state_dict()

        if epoch % 5 == 0:
            print(f"Епоха {epoch:03d} | Втрати: {loss.item():.4f} | Val AUC: {val_auc:.4f}")

    print("\nНавчання завершено.")
    model.load_state_dict(best_state)

            # Оцінемо найкращі моделі
    model.eval()
    with torch.no_grad():
        test_logits = model(X_test_t)
        test_probs = torch.sigmoid(test_logits).numpy()
        test_preds = (test_probs > 0.5).astype(int)

    print("\nРезультати на тестових даних:")
    print("Матриця плутанини:\n", confusion_matrix(y_test, test_preds))
    print("\nДетальний звіт:\n", classification_report(
        y_test, test_preds, digits=3, target_names=["Норма (0)", "Фрод (1)"]))
    print(f"ROC-AUC: {roc_auc_score(y_test, test_probs):.4f}")

except FileNotFoundError:
    print("Помилка: Файл 'HW-11.9_creditcard.csv' не знайдено.")
except Exception as e:
    print(f"Сталася неочікувана помилка: {e}")

Завантаження даних...
Попередня обробка даних...

Починаємо навчання нейронної мережі...
Епоха 005 | Втрати: 1.1821 | Val AUC: 0.9224
Епоха 010 | Втрати: 1.0678 | Val AUC: 0.9235
Епоха 015 | Втрати: 0.9835 | Val AUC: 0.9230
Епоха 020 | Втрати: 0.9065 | Val AUC: 0.9198
Епоха 025 | Втрати: 0.8485 | Val AUC: 0.9119
Епоха 030 | Втрати: 0.7920 | Val AUC: 0.9048
Епоха 035 | Втрати: 0.7402 | Val AUC: 0.9033
Епоха 040 | Втрати: 0.6807 | Val AUC: 0.9054
Епоха 045 | Втрати: 0.6343 | Val AUC: 0.9088
Епоха 050 | Втрати: 0.5777 | Val AUC: 0.9128

Навчання завершено.

Результати на тестових даних:
Матриця плутанини:
 [[41897   751]
 [   11    63]]

Детальний звіт:
               precision    recall  f1-score   support

   Норма (0)      1.000     0.982     0.991     42648
    Фрод (1)      0.077     0.851     0.142        74

    accuracy                          0.982     42722
   macro avg      0.539     0.917     0.566     42722
weighted avg      0.998     0.982     0.990     42722

ROC-AUC: 0.92